# 01 · Baseline y cuotas en validación (Persona B)

**Objetivo:** fijar los dos puntos de referencia contra los que se medirá todo modelo:
1. el **baseline de frecuencias** (el piso: no usa información del partido), y
2. las **cuotas de las casas de apuestas** (el techo práctico: usan toda la información del mercado).

Todo se evalúa en **validación (2023-24)**. El conjunto de prueba (2024-25) no se toca.

## 1. Carga y corte temporal
Usamos `split(df)` **sin** `use_test`: devuelve entrenamiento (2017-18 a 2022-23) y validación (2023-24).

In [ ]:
import sys; sys.path.append("..")
import pandas as pd
import numpy as np

from src.splits import split
from src.models.baseline import FrequencyBaseline
from src.evaluation.metrics import evaluate, compare, confusion, implied_probs
from src.evaluation.registro import guardar_resultado

df = pd.read_csv("../data/processed/matches.csv", parse_dates=["date"])
train, val = split(df)   # sin use_test: el test queda intacto
print(f"train: {len(train)} partidos ({train['date'].min():%Y-%m-%d} a {train['date'].max():%Y-%m-%d})")
print(f"val:   {len(val)} partidos ({val['date'].min():%Y-%m-%d} a {val['date'].max():%Y-%m-%d})")
assert train["date"].max() < val["date"].min()

## 2. Baseline de frecuencias
Se ajusta **solo con train**: predice siempre las proporciones históricas de H, D y A, sin mirar el partido.

In [ ]:
baseline = FrequencyBaseline().fit(None, train["result"])
print("Probabilidades fijas (H, D, A):", baseline.probs_.round(3))
p_base = baseline.predict_proba(val)
print("Frecuencias reales en validación:", val["result"].value_counts(normalize=True).reindex(["H", "D", "A"]).round(3).to_dict())

## 3. Probabilidades de mercado y partidos sin cuotas
`implied_probs()` convierte las cuotas en probabilidades y **normaliza** para quitar el margen de la casa. Las cuotas son solo referencia, nunca una feature del modelo.
Para que la comparación sea justa, ambos modelos se evalúan sobre **exactamente los mismos partidos** (los que tienen cuotas).

In [ ]:
odds_cols = ["odds_H", "odds_D", "odds_A"]
sin_cuotas = val[odds_cols].isna().any(axis=1)
print(f"Partidos de validación sin cuotas: {sin_cuotas.sum()} de {len(val)}")

val_ok = val[~sin_cuotas]
p_mercado = implied_probs(val_ok)
p_base_ok = baseline.predict_proba(val_ok)
assert len(val_ok) == len(p_mercado) == len(p_base_ok)
print("Partidos evaluados por ambos:", len(val_ok))

## 4. Margen promedio de la casa
Si las cuotas fueran justas, `1/odds_H + 1/odds_D + 1/odds_A` sumaría 1. El exceso sobre 1 es el margen (la ganancia de la casa).

In [ ]:
overround = 1 / val_ok["odds_H"] + 1 / val_ok["odds_D"] + 1 / val_ok["odds_A"]
margen = (overround - 1).mean()
print(f"Margen promedio de la casa en validación: {margen:.4f} ({margen:.2%})")

## 5. Comparación y matrices de confusión
Todas las métricas pasan por `evaluate()`. Menor log loss y Brier es mejor.

In [ ]:
y = val_ok["result"]
res_base = evaluate(y, p_base_ok, "baseline_frecuencias")
res_mercado = evaluate(y, p_mercado, "cuotas_mercado")
compare([res_base, res_mercado])

In [ ]:
print("Baseline de frecuencias"); display(confusion(y, p_base_ok))
print("Cuotas del mercado");       display(confusion(y, p_mercado))

## 6. Guardar resultados
Se guardan en `docs/resultados_validacion.csv` (una fila por modelo; si el modelo ya existe, se reemplaza). Los modelos siguientes usarán la misma función.

In [ ]:
guardar_resultado(res_base)
tabla = guardar_resultado(res_mercado)
tabla

## Conclusión
- El baseline de frecuencias (log loss ≈ 1.054, accuracy ≈ 46 %) es el piso: siempre predice "local gana", así que su accuracy es solo la frecuencia de victorias locales en validación.
- Las cuotas, tras quitar el margen (≈ 4 %), logran log loss ≈ 0.909 y Brier ≈ 0.533: reparten la probabilidad partido a partido según la fuerza de los equipos, y su accuracy sube a ≈ 59 %.
- Ninguno de los dos predice empates: la probabilidad de D (~20–30 %) casi nunca supera a la de H o A.
- Meta para los modelos propios: bajar del log loss del baseline y acercarse al de las cuotas. Nota: el rango ~0.97–1.00 de `CLAUDE.md` era una estimación; en 2023-24 las cuotas dan 0.909, y esa es la referencia real.